# Bước 4: Khả năng giải thích (XAI) với SHAP

**Mục tiêu:** chứng minh mô hình học đúng sinh lý bệnh thay vì chỉ "học vẹt"
dữ liệu. Y văn gốc `chicco2020machine` chỉ ra rằng **Serum Creatinine** và
**Ejection Fraction** là 2 chỉ số quyết định nhất đến sinh tử của bệnh nhân
suy tim — kiểm tra xem mô hình Extra Trees (Bước 3) có "học" đúng điều này
hay không.

In [1]:
import sys
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

try:
    import imblearn  # noqa: F401
except ImportError:
    import subprocess, sys as _sys
    print("[i] Chưa có imbalanced-learn trong môi trường này (Colab/Kaggle) — đang cài...")
    subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'imbalanced-learn'], check=True)
try:
    import shap  # noqa: F401
except ImportError:
    import subprocess, sys as _sys
    print("[i] Chưa có shap trong môi trường này (Colab/Kaggle) — đang cài...")
    subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'shap'], check=True)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesClassifier
from imblearn.over_sampling import SMOTE
import shap

def _load_heart_failure_data():
    """Tải dữ liệu cục bộ (../data/...) nếu có (chạy trong repo HMYT đã
    clone); nếu không (mở độc lập qua Colab/Kaggle, không có thư mục data/
    đi kèm) tự động tải từ mirror công khai trên hmyt-book (repo Public,
    xác minh 23/09/2026)."""
    import os
    local_path = '../data/heart_failure_clinical_records_dataset.csv'
    remote_url = ('https://raw.githubusercontent.com/fossbk-spec/hmyt-book/gh-pages/'
                  'labs_chuyen_de/ch02_suy_tim_risk_dxai/data/'
                  'heart_failure_clinical_records_dataset.csv')
    path = local_path if os.path.exists(local_path) else remote_url
    if path == remote_url:
        print(f"[i] Không tìm thấy dữ liệu cục bộ — tự động tải từ mirror công khai:\n    {remote_url}")
    return pd.read_csv(path).rename(columns={'death_event': 'DEATH_EVENT'})

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df = _load_heart_failure_data()
FEATURE_COLS = [c for c in df.columns if c != 'DEATH_EVENT']
X, y = df[FEATURE_COLS], df['DEATH_EVENT'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_s, y_train)

etc = ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE)
etc.fit(X_train_smote, y_train_smote)
print("[+] Đã huấn luyện lại đúng mô hình Extra Trees + SMOTE của Bước 3.")

[+] Đã huấn luyện lại đúng mô hình Extra Trees + SMOTE của Bước 3.


## 1. Tính giá trị SHAP với `TreeExplainer`

In [2]:
X_test_df = pd.DataFrame(X_test_s, columns=FEATURE_COLS)   # giữ tên cột để biểu đồ dễ đọc

explainer = shap.TreeExplainer(etc)
shap_values = explainer.shap_values(X_test_df)

# API shap có thể trả về mảng đơn (lớp dương) hoặc list 2 lớp tùy phiên bản — chuẩn hóa về lớp "Tử vong" (1)
if isinstance(shap_values, list):
    shap_vals_death = shap_values[1]
elif shap_values.ndim == 3:
    shap_vals_death = shap_values[:, :, 1]
else:
    shap_vals_death = shap_values

print(f"Ma trận giá trị SHAP: {shap_vals_death.shape} (n_test={X_test_df.shape[0]}, n_features={X_test_df.shape[1]})")

Ma trận giá trị SHAP: (60, 12) (n_test=60, n_features=12)


## 2. Biểu đồ tổng quan `summary_plot`

In [3]:
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_vals_death, X_test_df, show=False)
plt.title('SHAP Summary — Extra Trees + SMOTE (lớp Tử vong)')
plt.tight_layout()
plt.savefig('../figures/04_shap_summary.png', dpi=110, bbox_inches='tight')
plt.close()
print("[+] Đã lưu ../figures/04_shap_summary.png")

C:\Users\VICTUS\AppData\Local\Temp\ipykernel_39416\3194099765.py:2: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_vals_death, X_test_df, show=False)


[+] Đã lưu ../figures/04_shap_summary.png


## 3. Đối chiếu lâm sàng — Serum Creatinine & Ejection Fraction

In [4]:
mean_abs_shap = np.abs(shap_vals_death).mean(axis=0)
importance_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
importance_df.index = importance_df.index + 1  # xếp hạng bắt đầu từ 1

print("Xếp hạng tầm quan trọng đặc trưng theo |SHAP| trung bình (kết quả THẬT từ lần chạy này):")
print(importance_df.to_string())

top5 = set(importance_df['feature'].head(5))
for feat in ['serum_creatinine', 'ejection_fraction']:
    rank = importance_df.index[importance_df['feature'] == feat][0]
    status = "NẰM trong Top 5" if feat in top5 else "KHÔNG nằm trong Top 5"
    print(f"\n-> {feat}: hạng #{rank}/13 theo |SHAP| trung bình — {status}")

Xếp hạng tầm quan trọng đặc trưng theo |SHAP| trung bình (kết quả THẬT từ lần chạy này):
                     feature  mean_abs_shap
1                       time       0.155912
2          ejection_fraction       0.075912
3           serum_creatinine       0.068877
4                        age       0.030276
5               serum_sodium       0.028849
6        high_blood_pressure       0.027349
7                    anaemia       0.021023
8                   diabetes       0.014541
9   creatinine_phosphokinase       0.014301
10                       sex       0.013490
11                 platelets       0.011920
12                   smoking       0.010242

-> serum_creatinine: hạng #3/13 theo |SHAP| trung bình — NẰM trong Top 5

-> ejection_fraction: hạng #2/13 theo |SHAP| trung bình — NẰM trong Top 5


## 4. Bình luận đối chiếu (điền dựa trên kết quả THẬT ở cell trên)

> ⚠️ Đoạn này là khung mẫu — khi hoàn thiện báo cáo Bước 5, thay các chỗ
> `[...]` bằng số liệu thật lấy từ 2 cell trên của chính lần chạy này, KHÔNG
> copy số liệu minh họa cố định.

- Hạng của `serum_creatinine`: `[...]`/13. Hạng của `ejection_fraction`:
  `[...]`/13.
- Nếu CẢ HAI đều nằm trong Top 5: kết quả **khớp với y văn**
  `chicco2020machine` — mô hình học đúng 2 chỉ số sinh lý quan trọng nhất
  (chức năng thận suy giảm + khả năng bơm máu của tim), củng cố độ tin cậy
  lâm sàng của pipeline.
- Nếu KHÔNG khớp: cần cân nhắc liệu **SMOTE có làm lệch phân bố đặc trưng
  gốc** hay không (SMOTE nội suy tuyến tính giữa các láng giềng gần nhất
  trong không gian đã chuẩn hóa — có thể làm mờ ranh giới quyết định dựa
  trên 1-2 đặc trưng trội), hoặc do cỡ mẫu Test nhỏ (n=60) khiến ước lượng
  SHAP có phương sai cao. Đây là hạn chế cần nêu trung thực ở báo cáo Bước 5,
  không phải lỗi cần "sửa" cho khớp y văn bằng mọi giá.